In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import joblib
import warnings
from datetime import datetime

import pandas as pd
import numpy as np
import lightgbm as lgb

warnings.filterwarnings("ignore")

DATA_DIR = "/content/drive/MyDrive/NADAC"
OUTPUT_DIR = "/content/drive/MyDrive/LightGBM"

MODEL_PATH = os.path.join(OUTPUT_DIR, "lightgbm_model.pkl")
FEATURE_META_PATH = os.path.join(OUTPUT_DIR, "feature_metadata.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
csv_files = sorted([
    f for f in os.listdir(DATA_DIR)
    if f.endswith(".csv")
])

all_dfs = []

for file in csv_files:
    path = os.path.join(DATA_DIR, file)

    temp = pd.read_csv(
        path,
        usecols=["NDC", "NADAC Per Unit", "As of Date"],
        dtype={"NDC": str}
    )

    temp["NDC"] = temp["NDC"].str.zfill(11)
    all_dfs.append(temp)

raw_df = pd.concat(all_dfs, ignore_index=True)

print("Raw rows:", len(raw_df))

In [ ]:
raw_df["As of Date"] = pd.to_datetime(raw_df["As of Date"], errors="coerce")
raw_df["NADAC Per Unit"] = pd.to_numeric(raw_df["NADAC Per Unit"], errors="coerce")

raw_df = raw_df.dropna(subset=["NDC", "As of Date", "NADAC Per Unit"])

raw_df = raw_df.drop_duplicates(
    subset=["NDC", "As of Date"],
    keep="last"
)

raw_df["month"] = raw_df["As of Date"].dt.to_period("M").dt.to_timestamp("M")

panel = (
    raw_df
    .sort_values("As of Date")
    .groupby(["NDC", "month"], as_index=False)
    .last()
)

panel = panel[["NDC", "month", "NADAC Per Unit"]]
panel = panel.sort_values(["NDC", "month"]).reset_index(drop=True)

In [ ]:
MIN_RAW_MONTHS = 24

counts = panel.groupby("NDC")["month"].count()
valid_ndcs = counts[counts >= MIN_RAW_MONTHS].index

panel = panel[panel["NDC"].isin(valid_ndcs)].copy()
panel = panel.sort_values(["NDC", "month"]).reset_index(drop=True)

print("Valid NDC:", panel["NDC"].nunique())

In [ ]:
def fill_missing(group):
    group = group.sort_values("month").set_index("month")

    full_idx = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="ME"
    )

    group = group.reindex(full_idx)

    group["NDC"] = group["NDC"].ffill().bfill()
    group["NADAC Per Unit"] = group["NADAC Per Unit"].ffill()

    return group.reset_index().rename(columns={"index": "month"})

panel = (
    panel
    .groupby("NDC", group_keys=False)
    .apply(fill_missing)
    .reset_index(drop=True)
)

In [ ]:
LAGS = list(range(1, 13))

g = panel.groupby("NDC", group_keys=False)

for k in LAGS:
    panel[f"lag_{k}"] = g["NADAC Per Unit"].shift(k)

feat_cols = [f"lag_{k}" for k in LAGS]

panel["roll_mean_3"] = (
    panel.groupby("NDC")["NADAC Per Unit"]
    .transform(lambda s: s.shift(1).rolling(3).mean())
)

panel["roll_mean_6"] = (
    panel.groupby("NDC")["NADAC Per Unit"]
    .transform(lambda s: s.shift(1).rolling(6).mean())
)

panel["roll_std_6"] = (
    panel.groupby("NDC")["NADAC Per Unit"]
    .transform(lambda s: s.shift(1).rolling(6).std())
)

feat_cols += ["roll_mean_3", "roll_mean_6", "roll_std_6"]

panel["month_num"] = panel["month"].dt.month.astype(int)
feat_cols += ["month_num"]

In [ ]:
panel["next_price"] = g["NADAC Per Unit"].shift(-1)

panel["y_1"] = panel["next_price"] - panel["NADAC Per Unit"]

In [ ]:
train_df = panel.dropna(subset=feat_cols + ["y_1"]).copy()

X_train = train_df[feat_cols]
y_train = train_df["y_1"]

print("Train rows:", len(train_df))

In [ ]:
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=128,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained.")

In [ ]:
joblib.dump(model, MODEL_PATH)

meta = {
    "features": feat_cols,
    "lags": LAGS,
    "target": "delta",
    "created_at": datetime.utcnow().isoformat()
}

with open(FEATURE_META_PATH, "w") as f:
    json.dump(meta, f, indent=4)

print("Saved model.")